# Exercise 03 — Word Embeddings

Words are symbols, but neural networks only understand numbers. In this notebook you turn words into **vectors** that carry meaning, and then put those vectors to work on a real task.

## What you will do
1. **Train your own Word2Vec** model on the Brown corpus and inspect what it learned.
2. **Measure similarity** — implement cosine similarity by hand and check it against gensim.
3. **Explore pre-trained GloVe embeddings** — nearest neighbours, analogies and odd-one-out.
4. **Build a sentiment classifier** on top of GloVe embeddings, train it and evaluate it.
5. **Multiple-choice questions** — to check your understanding.

## How to work through it
- Run the cells **in order** and fill in **every `# TODO`**.
- Tasks are numbered (**1.1**, **1.2**, …). Tasks that say *Your answer here* want a short written answer, not code.
- Most implementation tasks are followed by a **✅ Check** cell that verifies your work automatically. Run it and make sure it passes before you move on.

> **Heads up — two cells download data.** The Brown corpus (~3 MB) and the GloVe vectors (~130 MB, a minute or two on a decent connection). Start them early so they are ready when you need them.


In [ ]:
import numpy as np
from typing import Any

import gensim.downloader
from gensim.models import Word2Vec
from gensim.utils import tokenize, simple_preprocess

import nltk
from nltk.corpus import brown, stopwords

from tqdm import tqdm
from datasets import load_dataset

import torch
from torch import nn
from torch.utils.data import DataLoader


## 1. Training your own Word2Vec model

To train **Word2Vec embeddings** we first need a text corpus. We use the **Brown Corpus**, a classic collection of American English texts from the 1960s that ships with NLTK.

We also grab NLTK's list of English **stopwords** (*the*, *and*, *is*, …). These words are extremely frequent but carry little meaning, so we filter them out before training and let the model spend its capacity on informative words.


In [ ]:
nltk.download("brown")
nltk.download("stopwords")

stop_words = set(stopwords.words("english"))
print(f"{len(stop_words)} stopwords, e.g. {sorted(stop_words)[:10]}")


### Exploring the corpus

Before training anything, look at the data.

**1.1 Fill in the three blanks below to get a feel for the corpus.**

Hints:
- `brown.sents()` returns the corpus as a list of sentences; each sentence is *already* a list of word strings.
- `brown.categories()` returns the genre labels (news, fiction, humor, …).
- To count all word **tokens**, sum the lengths of all the sentences.


In [ ]:
sentences = brown.sents()

n_sentences = len(sentences)
n_categories = len(brown.categories())
n_tokens = sum(len(s) for s in sentences)

print(f"Categories : {n_categories}  ->  {brown.categories()}")
print(f"Sentences  : {n_sentences:,}")
print(f"Word tokens: {n_tokens:,}")

print("\nFirst 3 sentences:")
for sent in sentences[:3]:
    print("  ", sent)


### Cleaning and preprocessing

Raw sentences need to be **preprocessed** before Word2Vec can use them. For every sentence we:

1. **Join** its words back into a single string — `" ".join(sent)`.
2. **Tokenize, lowercase and deaccent** it with `simple_preprocess(text, deacc=True, min_len=2)`. This also strips punctuation and drops words shorter than 2 characters.
3. **Remove stopwords**, keeping only tokens that are *not* in `stop_words`.

The result is a list of lists: one list of cleaned tokens per sentence — exactly the format `Word2Vec` expects.

**1.2 Build `cleaned_sentences` by applying the three steps above to every sentence in `sentences`.**

Hints:
- A nested list comprehension does this in one expression, but a plain `for` loop is just as good — write whichever you find clearer.
- The corpus has ~57k sentences, so wrap the loop in `tqdm(...)` if you want a progress bar.


In [ ]:
cleaned_sentences = [
    [w for w in simple_preprocess(" ".join(sent), deacc=True, min_len=2) if w not in stop_words]
    for sent in tqdm(sentences)
]

print("Raw    :", sentences[0])
print("Cleaned:", cleaned_sentences[0])


In [ ]:
# ✅ Check your preprocessing
assert isinstance(cleaned_sentences, list), "cleaned_sentences should be a list"
assert len(cleaned_sentences) == len(sentences), "one cleaned sentence per raw sentence"
assert all(isinstance(s, list) for s in cleaned_sentences[:100]), "each element should be a list of tokens"

flat = [w for s in cleaned_sentences for w in s]
assert not any(w in stop_words for w in flat[:5000]), "some stopwords survived the filter"
assert all(w == w.lower() for w in flat[:5000]), "tokens should be lowercased"

print(f"Tokens before cleaning: {sum(len(s) for s in sentences):,}")
print(f"Tokens after cleaning : {len(flat):,}")
print(f"Vocabulary size       : {len(set(flat)):,}")
print("Looks good ✅")


### Training the model

Now train the embeddings. The parameters that matter here:

- `vector_size` — how many dimensions each word vector has.
- `window` — how many words to the left and right count as "context".
- `min_count` — words rarer than this are dropped from the vocabulary.
- `sg` — `1` for **skip-gram** (predict context from the target word), `0` for **CBOW** (predict the target word from its context). Skip-gram works better on small corpora.
- `epochs` — how many passes over the data.

**1.3 Fill in the missing arguments: 50-dimensional vectors, a context window of 3, and the skip-gram objective.**

> Training takes a minute or two.


In [ ]:
w2v = Word2Vec(
    sentences=cleaned_sentences,
    vector_size=50,    # embedding size
    window=3,          # context window
    min_count=1,       # keep all words, even rare ones
    workers=4,         # number of CPU cores to use
    sg=1,              # skip-gram (better for small data)
    epochs=20,         # passes over the data
    seed=42,
)

print(f"Vocabulary size: {len(w2v.wv):,}")
print(f"Vector size    : {w2v.wv.vector_size}")


**1.4 Inspect what the model learned: look up the vector for a word, and find its nearest neighbours.**

Hints:
- `w2v.wv[word]` gives you the vector for a word.
- `w2v.wv.most_similar(word, topn=5)` returns the 5 closest words as `(word, similarity)` pairs.


In [ ]:
word = "jury"

vector = w2v.wv[word]
neighbours = w2v.wv.most_similar(word, topn=5)

print(f"Vector for '{word}' (first 10 of {len(vector)} dimensions):")
print(vector[:10])

print(f"\nMost similar to '{word}':")
for w, score in neighbours:
    print(f"   {w:<15} {score:.3f}")


**1.5 Try a few words of your own.** Add two or three words to the list below and look at their neighbours.


In [ ]:
for word in ["money", "school"]:  # TODO: add two or three words of your own
    if word in w2v.wv:
        print(f"{word:>12} -> {[w for w, _ in w2v.wv.most_similar(word, topn=5)]}")
    else:
        print(f"{word:>12} -> not in the vocabulary")


**1.6 Are the neighbours sensible?** The Brown Corpus is about 1 million tokens of 1960s American English. Which kinds of words would you expect this model to handle badly, and why?

---

*Answer:*

Partly. Some neighbours make sense — *school* → *schools*, *graduate*, *vocational*, *desegregation*; *jury* → *prosecution*, *subpenas* — but many are noise: surnames like *wexler* and *karns* for *jury*, or *hobbies* and *persuading* for *money*. (Your lists will differ a little: with `workers=4`, Word2Vec training is not bit-for-bit reproducible.)

Words this model handles badly:
- **Rare words.** About 38% of the 41k-word vocabulary occurs exactly once, and two thirds of it fewer than 5 times. With `min_count=1` such a word gets its vector from a handful of context windows, which is mostly noise. Rare words also *pollute* the neighbour lists of common words: *wexler* (6 occurrences) and *karns* (8) are simply names that appeared near *jury* in the same news stories.
- **Words from outside 1960s American English.** *internet* and *email* never occur, so they are out of vocabulary altogether, and words whose meaning has shifted since (*web*, *mouse*, *cloud*) only have their 1960s sense.
- **Topic-specific words.** Brown is a balanced sample of 500 texts across 15 genres, so any single domain (medicine, sports, law) gets only a few thousand tokens.
- **Words with several senses** (*bank*, *bass*, *spring*) get one vector that mixes all of them — true of every static embedding, but worse with little data.

The underlying issue is size: after cleaning, the model trains on ~530k tokens, compared to 6 billion for the GloVe vectors in Section 2.

---


### Cosine similarity

`most_similar` ranks words by **cosine similarity** — the cosine of the angle between two vectors:

\begin{align}
\cos(\mathbf{u}, \mathbf{v}) = \frac{\mathbf{u} \cdot \mathbf{v}}{\lVert \mathbf{u} \rVert \, \lVert \mathbf{v} \rVert}
\end{align}

It ignores the *length* of the vectors and only looks at their **direction**, which is what we want: the value is `1` for vectors pointing the same way, `0` for orthogonal ones, and `-1` for opposite ones.

**1.7 Implement `cosine_similarity` yourself.**

Hints:
- `np.dot(u, v)` for the dot product in the numerator.
- `np.linalg.norm(u)` for the length $\lVert \mathbf{u} \rVert$.


In [ ]:
def cosine_similarity(u: np.ndarray, v: np.ndarray) -> float:
    """Cosine similarity between two vectors."""
    return float(np.dot(u, v) / (np.linalg.norm(u) * np.linalg.norm(v)))


In [ ]:
# ✅ Check your implementation against gensim's own
for w1, w2 in [("jury", "court"), ("man", "woman"), ("jury", "car")]:
    mine = cosine_similarity(w2v.wv[w1], w2v.wv[w2])
    theirs = w2v.wv.similarity(w1, w2)
    assert np.isclose(mine, theirs, atol=1e-5), f"mismatch for ({w1}, {w2}): {mine} vs {theirs}"
    print(f"cos({w1:>5}, {w2:<6}) = {mine:+.4f}   (gensim: {theirs:+.4f})")

print("Looks good ✅")


**1.8 Which of the three pairs above is the most similar, and which the least? Does that match your intuition?**

---

*Answer:*

Most similar: **man–woman** (≈ 0.78). Least similar: **jury–car** (≈ 0.21), with **jury–court** (≈ 0.58) in between. These are the values from our run; yours will differ a little, since Word2Vec training is not bit-for-bit reproducible.

*jury–car* coming last matches intuition. What surprises many people is that *man–woman* beats *jury–court*: we think of *man* and *woman* as opposites, but they occur in almost the same contexts (*the ___ said*, *a young ___ who*), and the distributional hypothesis only looks at context. Cosine similarity between word vectors measures **relatedness of usage, not synonymy** — which is why antonyms also end up close together. GloVe shows the same pattern: *man–woman* 0.83 vs *jury–court* 0.75, and *good–bad* 0.77, *hot–cold* 0.73 (*cold* is the 2nd-nearest neighbour of *hot*).

Note also that even the unrelated pair is clearly positive rather than ≈ 0: in practice word vectors share a common overall direction, so cosine scores are best compared with each other rather than read on an absolute scale.

---


## 2. Pre-trained GloVe embeddings

Training on 1 million tokens gets you only so far. In practice we usually start from **pre-trained embeddings** learned on far more text.

Here we load **GloVe** (Global Vectors for Word Representation), trained on Wikipedia + Gigaword — about 6 billion tokens. Each word is mapped to a **100-dimensional** vector.

> The download is ~130 MB and is cached, so it is only slow the first time.


In [ ]:
glove_vectors = gensim.downloader.load("glove-wiki-gigaword-100")

print(f"Vocabulary size: {len(glove_vectors):,}")
print(f"Vector size    : {glove_vectors.vector_size}")


**2.1 Compare your own embeddings with GloVe's.** Print the 5 nearest neighbours of the same word according to each model.

Hint: `glove_vectors` has the same interface as `w2v.wv` — `most_similar`, `similarity`, and `glove_vectors[word]` all work.


In [ ]:
word = "jury"

print(f"Your Word2Vec (Brown, ~1M tokens):")
print("  ", [w for w, _ in w2v.wv.most_similar(word, topn=5)])

print(f"GloVe (Wikipedia + Gigaword, 6B tokens):")
print("  ", [w for w, _ in glove_vectors.most_similar(word, topn=5)])


**2.2 The two lists are different. Name two reasons why.** Think about *how much* text each model saw, and *what kind* of text it was.

---

*Answer:*

In our run, Word2Vec gave `wexler, reservations, subpenas, karns, prosecution` and GloVe gave `jurors, judge, trial, court, verdict`. GloVe's list is what you would find in a thesaurus; ours is a mix of legal words and noise.

1. **How much text.** Our model saw ~530k tokens after cleaning, and *jury* only 68 times; GloVe saw 6 billion tokens — about 10,000× more. With 68 examples, a word that happens to share one or two contexts with *jury* (a surname like *wexler*, or *subpenas*, which occurs exactly once) can end up as a top neighbour. With billions of tokens those accidents average out and only the consistent associations — *judge*, *trial*, *verdict* — survive.
2. **What kind of text.** Brown is 1960s American English spread across 15 genres. Two thirds of its uses of *jury* are in the news section, and 18 come from a single report on a Fulton County grand-jury investigation — the very first document, which you printed in Task 1.1. The neighbours reflect those particular stories, including the period spelling *subpenas*. GloVe is trained on present-day Wikipedia and newswire, which covers juries and courts from every angle.

A third, smaller factor is the method itself: GloVe factorises global co-occurrence counts with 100-dimensional vectors, whereas we trained 50-dimensional skip-gram vectors on stopword-filtered text.

---


### Analogies

The famous property of word embeddings is that **directions in the vector space carry meaning**. The vector that takes you from *man* to *king* is roughly the same one that takes you from *woman* to *queen*:

$$\text{king} - \text{man} + \text{woman} \approx \text{queen}$$

`most_similar` can do this arithmetic for you: words in `positive=[...]` are added, words in `negative=[...]` are subtracted.

**2.3 Implement `analogy(a, b, c)` — read as "*a* is to *b* as *c* is to ...?" — which computes $b - a + c$.**

Hint: that is `most_similar(positive=[b, c], negative=[a], topn=topn)`.


In [ ]:
def analogy(a: str, b: str, c: str, topn: int = 3) -> list:
    """a is to b as c is to ...?   ->   b - a + c"""
    return glove_vectors.most_similar(positive=[b, c], negative=[a], topn=topn)


print("man   -> king   ::  woman ->", [w for w, _ in analogy("man", "king", "woman")])
print("paris -> france ::  tokyo ->", [w for w, _ in analogy("paris", "france", "tokyo")])
print("good  -> better ::  bad   ->", [w for w, _ in analogy("good", "better", "bad")])
print("big   -> bigger ::  small ->", [w for w, _ in analogy("big", "bigger", "small")])


### Odd one out

Since we can measure similarity, we can also spot the word that does not belong: `doesnt_match` returns the word furthest from the average of the group.

**2.4 Complete the loop and add a group of your own.**


In [ ]:
groups = [
    ["breakfast", "lunch", "dinner", "football"],
    ["denmark", "sweden", "norway", "guitar"],
    ["red", "green", "blue", "monday"],
    # GloVe answers "rust", not "banana": in its 2014 Wikipedia/news text,
    # "rust" mostly means corrosion, not the programming language.
    ["python", "java", "rust", "banana"],
]

for group in groups:
    odd = glove_vectors.doesnt_match(group)
    print(f"{str(group):<50} -> {odd}")


### A word of caution

Embeddings learn whatever regularities are present in their training text — including social stereotypes. Run the cell below and look at what comes out.


In [ ]:
print("man -> doctor     ::  woman ->", [w for w, _ in analogy("man", "doctor", "woman", topn=5)])
print("man -> programmer ::  woman ->", [w for w, _ in analogy("man", "programmer", "woman", topn=5)])
print("man -> boss       ::  woman ->", [w for w, _ in analogy("man", "boss", "woman", topn=5)])


**2.5 What do you see, and where does it come from?** Why is this a problem if these embeddings are used as features in, say, a CV-screening system?

*(We come back to this in the Responsible AI session.)*

---

*Answer:*

In our run:
- *man → doctor :: woman → ?* gives **nurse** as the top answer — ahead of *physician*.
- *man → boss :: woman → ?* gives *bosses*, then **girlfriend**, *boyfriend*, *colleague*, *lover* — relationship words rather than words about authority.
- *man → programmer :: woman → ?* is milder here (*educator*, *programmers*, *linguist*, *technician*); the famous *computer programmer → homemaker* result (Bolukbasi et al., 2016) came from word2vec trained on Google News.

**Where it comes from:** nothing in GloVe was programmed to be sexist. The vectors reflect the co-occurrence statistics of Wikipedia and news text, and that text describes men and women differently. You can measure the asymmetry directly: *doctor* is about equally close to *man* and *woman* (cosine 0.61 vs 0.63), but *nurse* is much closer to *woman* than to *man* (0.61 vs 0.46). The embedding faithfully encodes how people *wrote*, including stereotypes and historical imbalances.

**A caveat:** `most_similar` never returns one of its input words, so the analogy is not allowed to answer *doctor*. Rank *all* words and $\text{doctor} - \text{man} + \text{woman}$ is in fact closest to *doctor* itself (and likewise *boss* and *programmer* for the other two). Analogy tests therefore exaggerate bias somewhat. The bias itself is still real: that *nurse* ranks above *physician* is not an artefact.

**Why this matters for CV screening:** a model that scores CVs from these features will inherit the associations. Words and names associated with women would sit closer to *nurse*, *assistant* or *receptionist* and further from *engineer* or *manager*, so two CVs that differ only in gendered words could get different scores. That discriminates against candidates at scale, invisibly — no one wrote a rule saying "prefer men", so the bias is hard to notice and to audit. It can also be illegal under equal-treatment law. Amazon scrapped an experimental hiring model in 2018 for exactly this kind of behaviour.

---


## 3. Using GloVe embeddings to train a classifier

So far we have looked at individual words. Now we use embeddings as **features** for a downstream task: classifying the sentiment of tweets.

We use the **Sentiment140** dataset, where each tweet is labelled `1` (positive) or `0` (negative). The full dataset has 1.6M tweets; we take a subset to keep things fast.


In [ ]:
ds = load_dataset("adilbekovich/Sentiment140Twitter")

train = ds["train"].select(range(50_000))
test = ds["test"].select(range(10_000))

print(f"Training set size: {len(train):,}")
print(f"Test set size    : {len(test):,}")

print("\nA few examples:")
for row in train.select(range(3)):
    print(f"  label={row['label']}  {row['text'][:90]}")


### From a sentence to a vector

A neural network needs a **fixed-length** vector per example, but tweets have different lengths. The simplest fix is to **average the word vectors** of every word in the sentence.

It is a crude representation — it throws away word order completely — but it is a surprisingly strong baseline.

**3.1 Implement `sentence_embedding`.**

Steps:
1. **Tokenize** the sentence: `list(tokenize(sentence, deacc=True, to_lower=True))`.
2. **Look up** the vector of every token the model knows. Use `w in model` to test membership — tweets are full of typos and hashtags that GloVe has never seen.
3. **Average** the vectors with `np.mean(..., axis=0)`. If *no* word was known, return a **zero vector** of length `model.vector_size` instead (averaging an empty list would fail).


In [ ]:
def sentence_embedding(sentence: str, model: Any) -> np.ndarray:
    """Average the embeddings of all known words in `sentence`."""
    tokens = list(tokenize(sentence, deacc=True, to_lower=True))
    word_vectors = [model[w] for w in tokens if w in model]
    if not word_vectors:
        return np.zeros(model.vector_size, dtype=np.float32)
    return np.mean(word_vectors, axis=0)


In [ ]:
# ✅ Check your sentence_embedding
v = sentence_embedding("i love this movie", glove_vectors)
assert v.shape == (100,), f"expected shape (100,), got {v.shape}"
expected = np.mean([glove_vectors[w] for w in ["i", "love", "this", "movie"]], axis=0)
assert np.allclose(v, expected, atol=1e-5), "should be the *average* of the word vectors"

unknown = sentence_embedding("zzzzqqq wwwwxyzzz", glove_vectors)
assert unknown.shape == (100,) and np.allclose(unknown, 0), "unknown-only sentences should give a zero vector"

# Sentences with a similar meaning should end up closer together
a = sentence_embedding("the food was delicious", glove_vectors)
b = sentence_embedding("the meal tasted great", glove_vectors)
c = sentence_embedding("my laptop battery died", glove_vectors)
print(f"similar pair  : {cosine_similarity(a, b):.3f}")
print(f"unrelated pair: {cosine_similarity(a, c):.3f}")
assert cosine_similarity(a, b) > cosine_similarity(a, c)

print("Looks good ✅")


**3.2 Add an `"embeddings"` column to both splits by applying `sentence_embedding` to the `"text"` column.**

Hints:
- `dataset.map(fn)` applies `fn` to every row; `fn` receives the row as a dict and returns a dict of **new** columns.
- So: `train.map(lambda x: {"embeddings": sentence_embedding(x["text"], glove_vectors)})`.
- `set_format(type="torch", ...)` is already written for you — it makes the dataset hand back PyTorch tensors.

> This takes a minute or so for 60k tweets.


In [ ]:
train = train.map(lambda x: {"embeddings": sentence_embedding(x["text"], glove_vectors)})
test = test.map(lambda x: {"embeddings": sentence_embedding(x["text"], glove_vectors)})

train.set_format(type="torch", columns=["embeddings", "label"])
test.set_format(type="torch", columns=["embeddings", "label"])

print(train)


### The classifier

A small feed-forward network — the same building blocks you implemented by hand last week:

1. **Input**: the 100-dimensional averaged embedding of a tweet.
2. **Hidden layer**: `Linear(input_dim, 256)` followed by a **ReLU**.
3. **Output layer**: `Linear(256, 1)` followed by a **Sigmoid**, so the output is a probability in $(0, 1)$ — the probability that the tweet is positive.

**3.3 Fill in the layers and the forward pass, then create the model with the right input dimension.**

Hint: the input dimension is the size of a GloVe vector — `glove_vectors.vector_size`.


In [ ]:
class SentimentClassifier(nn.Module):
    def __init__(self, input_dim: int, hidden_dim: int = 256):
        super().__init__()
        self.fc1 = nn.Linear(input_dim, hidden_dim)
        self.relu = nn.ReLU()
        self.fc2 = nn.Linear(hidden_dim, 1)
        self.sigmoid = nn.Sigmoid()

    def forward(self, x):
        x = self.fc1(x)
        x = self.relu(x)
        x = self.fc2(x)
        x = self.sigmoid(x)
        return x


classifier = SentimentClassifier(input_dim=glove_vectors.vector_size)


In [ ]:
# ✅ Check your classifier
dummy = torch.randn(8, glove_vectors.vector_size)
out = classifier(dummy)

assert out.numel() == 8, f"expected one output per example, got shape {tuple(out.shape)}"
assert (out >= 0).all() and (out <= 1).all(), "outputs should be probabilities — did you apply the sigmoid?"

print(classifier)
print(f"\nTrainable parameters: {sum(p.numel() for p in classifier.parameters()):,}")
print("Looks good ✅")


### Training setup

- **Batch size 64** — how many tweets we process before each parameter update.
- **30 epochs** — how many times we go through the whole training set.
- **`BCELoss`** — binary cross-entropy, the standard loss when the model outputs a single probability.
- **SGD, lr = 0.01** — the optimizer that applies the updates.


In [ ]:
BATCH_SIZE = 64
EPOCHS = 30

loss_fn = nn.BCELoss()
optimizer = torch.optim.SGD(classifier.parameters(), lr=0.01)

train_dataloader = DataLoader(train, batch_size=BATCH_SIZE, shuffle=True)
test_dataloader = DataLoader(test, batch_size=BATCH_SIZE)


**3.4 Write a small `accuracy` helper.** The model outputs probabilities, but accuracy needs hard 0/1 decisions.

Hints:
- Threshold at 0.5: `(outputs > 0.5)` gives a boolean tensor — `.float()` turns it into 0s and 1s.
- Then compare with `labels` and take the **mean** of the matches.


In [ ]:
def accuracy(outputs: torch.Tensor, labels: torch.Tensor) -> float:
    """Fraction of correct predictions. `outputs` are probabilities in (0, 1)."""
    predictions = (outputs > 0.5).float()
    return (predictions == labels).float().mean().item()


In [ ]:
# ✅ Check your accuracy function
probs = torch.tensor([0.9, 0.2, 0.6, 0.4])
labels = torch.tensor([1.0, 0.0, 0.0, 0.0])
assert np.isclose(float(accuracy(probs, labels)), 0.75), f"expected 0.75, got {accuracy(probs, labels)}"
print("Looks good ✅")


### Training the classifier

The loop is the same one you saw last week. For each batch:

1. **Forward pass** — run the embeddings through the model.
2. **Compute the loss** — how far the predictions are from the labels.
3. **Backward pass** — `zero_grad()`, `backward()`, `step()`.

**3.5 Fill in the three steps.**

Hint: the model returns a tensor of shape `(batch_size, 1)` while the labels have shape `(batch_size,)`. Use `.squeeze()` on the outputs so `BCELoss` gets matching shapes.

> Training 30 epochs over 50k tweets takes around 10 minutes on a CPU. Start it and read ahead.


In [ ]:
classifier.train()
for epoch in range(EPOCHS):
    epoch_loss = 0.0

    for batch in train_dataloader:
        inputs = batch["embeddings"].float()
        labels = batch["label"].float()

        # 1. forward pass
        outputs = classifier(inputs)

        # 2. compute the loss
        loss = loss_fn(outputs.squeeze(), labels)

        # 3. backward pass and optimization
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        epoch_loss += loss.item()

    print(f"Epoch {epoch + 1:>2}/{EPOCHS}, Loss: {epoch_loss / len(train_dataloader):.4f}")


### Evaluation

**3.6 Evaluate the trained model on the test set.**

Remember to:
- put the model in **evaluation mode**,
- run the loop inside **`torch.no_grad()`** — no gradients are needed here,
- accumulate the loss and the accuracy per batch, then divide by the number of **batches**.


In [ ]:
classifier.eval()
total_loss, total_acc = 0.0, 0.0

with torch.no_grad():
    for batch in test_dataloader:
        inputs = batch["embeddings"].float()
        labels = batch["label"].float()

        outputs = classifier(inputs).squeeze()
        total_loss += loss_fn(outputs, labels).item()
        total_acc += accuracy(outputs, labels)

print(f"Test loss    : {total_loss / len(test_dataloader):.4f}")
print(f"Test accuracy: {100 * total_acc / len(test_dataloader):.2f}%")


### Try it on your own sentences

**3.7 Write `predict_sentiment(text)`, which returns the probability that `text` is positive.**

Hints:
- Embed the text with `sentence_embedding(text, glove_vectors)`.
- Turn it into a tensor: `torch.tensor(vec).float()`.
- The model expects a **batch**, so add a leading dimension with `.unsqueeze(0)`.
- Wrap the call in `torch.no_grad()` and use `.item()` to get a plain float back.


In [ ]:
def predict_sentiment(text: str) -> float:
    """Probability that `text` expresses positive sentiment."""
    vec = torch.tensor(sentence_embedding(text, glove_vectors)).float().unsqueeze(0)
    classifier.eval()
    with torch.no_grad():
        return classifier(vec).item()


for text in [
    "i love this, best day ever",
    "worst service i have ever had, never coming back",
    "the weather is nice today",
    "this is absolutely terrible",
    "thanks, that really helped",
]:
    print(f"{predict_sentiment(text):.2f}   {text}")


### Where this representation breaks down

Averaging word vectors throws away **word order**. The two sentences below contain exactly the same words.


In [ ]:
a = "the movie was good, not bad at all"
b = "the movie was bad, not good at all"

print(f"{predict_sentiment(a):.3f}   {a}")
print(f"{predict_sentiment(b):.3f}   {b}")


**3.8 Explain what you see.** Why does the model give these two sentences (almost) the same score, and what would a model need in order to tell them apart?

---

*Answer:*

The two scores are the same — 0.368 for both in our run — and that is no coincidence. After tokenization both sentences contain exactly the same eight words (*the, movie, was, good, not, bad, at, all*), only in a different order. Averaging ignores order, so both sentences get the same embedding (up to floating-point rounding, ~$10^{-7}$), and the classifier *has* to give them the same score. Both land on the negative side of 0.5, so the model gets the first sentence wrong. No amount of extra training can fix that: the information needed to separate them is gone before the classifier ever sees the input.

What the average throws away is exactly what decides the meaning here: **word order and composition**. *not* flips the meaning of the word that follows it, which is *bad* in one sentence and *good* in the other. Blind spots like this are part of why the classifier levels off at about 71% test accuracy — decent for averaged word vectors, but negation, sarcasm and contrast ("*the food was good but the service was terrible*") are invisible to it.

To tell the two sentences apart, a model has to read the words **in sequence** and let each word's representation depend on its neighbours:
- the cheapest fix is **n-gram features** — the bigrams *not bad* and *not good* are different features;
- a **recurrent network (RNN / LSTM)** reads the sentence left to right and carries a running state (next week);
- a **transformer** uses self-attention plus position information, so *not* can directly modify the word after it (week 5). The result is **contextual embeddings** — see MCQ 4.6.

---


## 4. MCQ

Answer each question by writing the letter of your choice (A–D) after **Answer:**.

---

### 4.1. Purpose of Word Embeddings

What is the main purpose of word embeddings in NLP?

A. To convert words into high-dimensional one-hot vectors<br>
B. To map words into continuous vector spaces that capture semantic meaning<br>
C. To remove stopwords from text before processing<br>
D. To reduce the training time of convolutional networks<br>

**Answer:** B ✅

---

### 4.2. One-Hot vs. Embeddings

Compared to one-hot encoding, word embeddings:

A. Have the same dimensionality as the vocabulary size<br>
B. Provide dense, low-dimensional representations that capture similarities<br>
C. Are always manually designed by experts<br>
D. Cannot be trained with neural networks<br>

**Answer:** B ✅

---

### 4.3. Word2Vec Models

The Skip-gram model in Word2Vec is designed to:

A. Predict the context words given a target word<br>
B. Predict the target word given the context words<br>
C. Cluster words into fixed categories<br>
D. Remove rare words from the corpus<br>

**Answer:** A ✅

---

### 4.4. Embedding Matrix Shape

In a neural network with vocabulary size $V$ and embedding dimension $d$, the embedding matrix has shape:

A. $(d \times V)$ <br>
B. $(V \times d)$<br>
C. $(V \times V)$<br>
D. $(d \times d)$<br>

**Answer:** B ✅

---

### 4.5. Semantic Relationships

Word embeddings can capture analogies such as:

A. king – man + woman ≈ queen<br>
B. dog – cat + car ≈ airplane<br>
C. apple – red + fast ≈ running<br>
D. chair – table + sky ≈ cloud<br>

**Answer:** A ✅

---

### 4.6. Contextual vs. Static Embeddings

How do contextual embeddings (e.g., BERT) differ from static embeddings (e.g., Word2Vec)?

A. They assign the same vector to a word regardless of context<br>
B. They assign different vectors to a word depending on its context<br>
C. They are always lower-dimensional than static embeddings<br>
D. They do not require pretraining on large corpora<br>

**Answer:** B ✅

---

### 4.7. Sparse vs. Dense Representations

Compared to Bag-of-Words (BoW) vectors, neural word embeddings are:

A. Higher dimensional and sparse<br>
B. Always binary representations<br>
C. Lower dimensional and sparse<br>
D. Lower dimensional and dense<br>

**Answer:** D ✅

---

### 4.8. Cosine Similarity

Why is cosine similarity, rather than Euclidean distance, the usual way to compare two word vectors?

A. It is the only similarity measure that can be computed efficiently<br>
B. It compares the direction of the vectors and ignores their magnitude<br>
C. It always returns a value between 0 and 1<br>
D. It requires the vectors to be normalised in advance<br>

**Answer:** B ✅

---

### 4.9. CBOW vs. Skip-gram

The Continuous Bag of Words (CBOW) model aims to:

A. Predict the target word given its surrounding context words<br>
B. Assign unique one-hot vectors to words<br>
C. Predict the context words given a target word<br>
D. Cluster words into topics using SVD<br>

**Answer:** A ✅

---

### 4.10. Distributional Semantics

The idea that "you shall know a word by the company it keeps" refers to:

A. Overfitting in NLP models<br>
B. Context-based learning of embeddings<br>
C. Sentence segmentation<br>
D. Stopword removal<br>

**Answer:** B ✅
